# Hex6 Colab VS Code Launch

Use this notebook with the Google Colab VS Code extension or in the Colab browser UI.

It is designed for:
- mounting Drive
- ensuring the repo exists under `/content/drive/MyDrive/Hex-A-Toe`
- installing the repo
- checking the GPU tier
- launching the strongest current cycle lane or the runtime benchmark lane


## Important

This notebook runs whatever repo contents are currently in Drive.

If your local workspace has newer changes than GitHub, push them first or copy the updated repo into Drive before running long jobs.


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
import pathlib
import subprocess

REPO_DIR = pathlib.Path('/content/drive/MyDrive/Hex-A-Toe')
GIT_URL = 'https://github.com/Stroudmj00/hex6-bot.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', GIT_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(os.getcwd())


/content/drive/MyDrive/Hex-A-Toe


In [7]:
subprocess.run(['python', '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[dev]'], check=True)


CompletedProcess(args=['python', '-m', 'pip', 'install', '-e', '.[dev]'], returncode=0)

In [8]:
import torch
print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


cuda: False
gpu: cpu


In [11]:
import torch
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


cuda: False
gpu: cpu


## Optional secrets and tracking

If you use Colab secrets, run this cell. Otherwise skip it.


In [9]:
import os
try:
    from google.colab import userdata
    token = userdata.get('HEX6_GITHUB_TOKEN')
    if token:
        os.environ['HEX6_GITHUB_TOKEN'] = token
        print('Loaded HEX6_GITHUB_TOKEN from Colab secrets.')
except Exception as exc:
    print(f'No HEX6_GITHUB_TOKEN loaded: {exc}')

try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
    if wandb_key:
        os.environ['WANDB_API_KEY'] = wandb_key
        os.environ['HEX6_ENABLE_WANDB'] = '1'
        os.environ['HEX6_WANDB_MODE'] = 'online'
        os.environ['HEX6_WANDB_PROJECT'] = 'hex6-bot'
        os.environ['HEX6_WANDB_TAGS'] = 'colab,cycle'
        os.environ['WANDB_DIR'] = '/content/drive/MyDrive/Hex-A-Toe/artifacts/wandb'
        print('Configured W&B tracking.')
except Exception as exc:
    print(f'No WANDB_API_KEY loaded: {exc}')


No HEX6_GITHUB_TOKEN loaded: Requesting secret HEX6_GITHUB_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.
No WANDB_API_KEY loaded: Requesting secret WANDB_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.


## Strongest current cycle run

This is the main training lane. It requires at least a `V100`.


In [10]:
subprocess.run([
    'python',
    'scripts/colab_run.py',
    'cycle',
    '--repo-root', '/content/drive/MyDrive/Hex-A-Toe',
    '--minimum-gpu-tier', 'V100',
    '--config', 'configs/colab_strongest_v2.toml',
    '--output-root', 'artifacts/bootstrap_colab_strongest_v2',
    '--minutes', '60',
    '--start-checkpoint', 'artifacts/alphazero_cycle_4h_strongest_v2/cycle_002/bootstrap_model.pt',
    '--status-backend', 'github_branch',
], check=True)


CalledProcessError: Command '['python', 'scripts/colab_run.py', 'cycle', '--repo-root', '/content/drive/MyDrive/Hex-A-Toe', '--minimum-gpu-tier', 'V100', '--config', 'configs/colab_strongest_v2.toml', '--output-root', 'artifacts/bootstrap_colab_strongest_v2', '--minutes', '60', '--start-checkpoint', 'artifacts/alphazero_cycle_4h_strongest_v2/cycle_002/bootstrap_model.pt', '--status-backend', 'github_branch']' returned non-zero exit status 2.

## Efficiency benchmark sweep

Run this instead of the cycle cell when the goal is throughput tuning.


In [ ]:
subprocess.run([
    'python',
    'scripts/colab_run.py',
    'runtime-benchmark',
    '--repo-root', '/content/drive/MyDrive/Hex-A-Toe',
    '--minimum-gpu-tier', 'V100',
    '--config', 'configs/colab_strongest_v2.toml',
    '--output', 'artifacts/runtime_parallelism_colab',
    '--cpu-threads', '8',
    '--interop-threads', '2',
    '--self-play-workers', '4', '8', '12',
    '--data-loader-workers', '2',
    '--parallel-expansions-per-root', '4', '6', '8',
    '--root-simulations', '96',
    '--bootstrap-games', '2',
    '--epochs', '1',
    '--max-game-plies', '0',
], check=True)
